# 05 · ESP32 measurements: tables and figures

Reads the Serial Monitor logs saved from the board (`esp32/serial_logs/<dataset>_serial.txt`) and produces
* the hardware results table (agreement, latency distribution, flash, heap),
* per-class agreement (does every class path of the tree match on the device?),
* figures `paper/figures/hw_*.png|pdf` and tables `paper/tables/hw_*.csv|tex`.

No raw dataset and no trained model are needed.

### Log formats
| | Benchmark sketch (`esp32_dt_test.ino`, notebook 03) | Legacy sketch |
|---|---|---|
| prediction agreement | yes | yes |
| inference latency | timed around `predict_esp32_dt()` only; min / median / p95 / p99 / max | no: the timer included `Serial.print`, so the value is serial-bound |
| flash / heap / chip info | yes (`#META` lines) | from the compiler output only |

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "paper_utils.py").exists():
    if ROOT.parent == ROOT: raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
from IPython.display import display
from paper_utils import *
set_paper_style()
LOGS = ROOT / "esp32" / "serial_logs"
FIG, TAB = ROOT / "paper" / "figures", ROOT / "paper" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
print("log folder:", LOGS, "|", [p.name for p in LOGS.glob("*.txt")] or "no logs yet")

## 1 · Parse every available log

In [ ]:
parsed, rows = {}, []
for ds in DATASETS:
    p = LOGS / f"{ds}_serial.txt"
    if not p.exists():
        print(f"  {DS_LABEL[ds]:18s}: no log yet ({p.name})"); continue
    log = parse_serial_log(p)
    if log["samples"].empty:
        print(f"  {DS_LABEL[ds]:18s}: log found but no sample rows parsed -- check the file"); continue
    parsed[ds] = log
    rows.append(summarize_hardware(ds, log))
    print(f"  {DS_LABEL[ds]:18s}: {len(log['samples'])} samples | format: {rows[-1]['log_format']} | complete={log['complete']}")

hw = pd.DataFrame(rows)
display(hw.T if len(hw) else "no hardware logs parsed yet")

## 2 · Python ↔ ESP32 agreement (implementation fidelity)

In [ ]:
if parsed:
    agree_rows = []
    for ds, log in parsed.items():
        df = log["samples"].copy()
        df["agree"] = df["esp32_pred"] == df["python_pred"]
        g = df.groupby("true_label").agg(n=("agree", "size"), matches=("agree", "sum")).reset_index()
        g["dataset"] = ds; agree_rows.append(g)
    pc = pd.concat(agree_rows)
    pc["agreement"] = pc["matches"] / pc["n"]
    display(pc.pivot(index="true_label", columns="dataset", values="agreement"))
    save_table(pc[["dataset", "true_label", "n", "matches", "agreement"]], TAB / "hw_per_class_agreement",
               caption="Python vs ESP32 prediction agreement per true class.", label="tab:hw_perclass")

    fig, ax = plt.subplots(figsize=(6, 3.2))
    labels = [DS_LABEL[d] for d in parsed]
    m = [int(hw.loc[hw.dataset == d, "python_esp32_matches"].iloc[0]) for d in parsed]
    n = [int(hw.loc[hw.dataset == d, "n_samples"].iloc[0]) for d in parsed]
    ax.bar(labels, [a / b * 100 for a, b in zip(m, n)], color=[DS_COLOR[d] for d in parsed])
    for i, (a, b) in enumerate(zip(m, n)): ax.text(i, 100.4, f"{a}/{b}", ha="center", fontsize=9)
    ax.set_ylim(95, 101.5); ax.set_ylabel("Python–ESP32 agreement (%)")
    ax.set_title("Implementation fidelity on the physical ESP32")
    save_fig(fig, FIG / "hw_agreement"); plt.show()
else:
    print("(nothing to plot yet)")

## 3 · Inference latency (benchmark-sketch logs only)

Reported as median and p95/p99 as well as the mean, because a single slow sample (cache miss, interrupt) can move a mean.
`mean_latency_us_batch` is the mean over `BATCH_REPS` repeated passes (the 1 µs timer resolution would otherwise dominate for small trees).

In [ ]:
timed = {ds: log for ds, log in parsed.items() if not log["legacy"]}
if timed:
    cols = ["dataset", "n_samples", "mean_latency_us_batch", "median_us", "p95_us", "p99_us", "max_us", "throughput_samples_per_s"]
    lat = hw[hw["dataset"].isin(timed)][cols].copy(); lat["dataset"] = lat["dataset"].map(DS_LABEL)
    display(lat)
    save_table(lat, TAB / "hw_latency", caption="On-device inference latency of predict\\_esp32\\_dt() (microseconds per sample).",
               label="tab:hw_latency", float_fmt="{:.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for ds, log in timed.items():
        axes[0].hist(log["samples"]["infer_us"], bins=60, alpha=.6, label=DS_LABEL[ds], color=DS_COLOR[ds])
    axes[0].set_xlabel("inference time per sample (µs)"); axes[0].set_ylabel("samples"); axes[0].set_yscale("log"); axes[0].legend()
    axes[0].set_title("per-sample latency distribution")
    x = np.arange(len(timed)); w = .2
    for j, (c, lab) in enumerate([("median_us", "median"), ("p95_us", "p95"), ("p99_us", "p99"), ("max_us", "max")]):
        axes[1].bar(x + (j - 1.5) * w, [float(hw.loc[hw.dataset == d, c].iloc[0]) for d in timed], w, label=lab)
    axes[1].set_xticks(x, [DS_LABEL[d] for d in timed]); axes[1].set_ylabel("µs"); axes[1].legend()
    axes[1].set_title("latency percentiles")
    save_fig(fig, FIG / "hw_latency"); plt.show()

    # latency vs tree depth along the executed path is not logged; show vs node count instead
    if "n_nodes" in hw.columns:
        fig, ax = plt.subplots(figsize=(4.8, 3.4))
        for d in timed:
            r = hw[hw.dataset == d].iloc[0]
            ax.scatter(float(r["n_nodes"]), float(r["mean_latency_us_batch"]), s=70, color=DS_COLOR[d], label=DS_LABEL[d])
        ax.set_xlabel("tree nodes"); ax.set_ylabel("mean latency (µs)"); ax.set_xscale("log"); ax.legend()
        save_fig(fig, FIG / "hw_latency_vs_nodes"); plt.show()
else:
    print("No benchmark-sketch logs yet -> no inference latency to report.\n"
          "Flash the sketch produced by notebook 03 and save its Serial output as esp32/serial_logs/<dataset>_serial.txt")

## 4 · Why the legacy latency is serial-bound

At 115 200 baud (8N1) every character costs about 86.8 µs. The legacy sketch printed one line per sample inside the timed region.
This cell compares the legacy µs/sample with the time needed merely to transmit those lines.

In [ ]:
leg = pd.read_csv(ROOT / "esp32" / "legacy_reported_measurements.csv")
us_per_char = 10 / 115200 * 1e6
out = []
for _, r in leg.iterrows():
    f = ROOT / "results" / "esp32_export" / r["dataset"] / "python_expected_predictions.csv"
    if not f.exists(): continue
    e = pd.read_csv(f, usecols=["sample_id", "true_label", "python_pred"]).head(int(r["n_samples"]))
    chars = (e.sample_id.astype(str).str.len() + e.true_label.astype(str).str.len() + 2 * e.python_pred.astype(str).str.len() + 5 + 2)  # ids,true,py,esp,match + CRLF
    est = chars.mean() * us_per_char
    out.append({"dataset": DS_LABEL[r["dataset"]], "legacy_reported_us_per_sample": r["us_per_sample_reported"],
                "UART_time_for_the_printed_line_us": round(est, 1), "ratio_reported_to_UART": round(r["us_per_sample_reported"] / est, 3)})
leg_cmp = pd.DataFrame(out); display(leg_cmp)
if len(leg_cmp): save_table(leg_cmp, TAB / "hw_legacy_latency_diagnosis", caption="Legacy latency vs UART transmission time.", label="tab:hw_legacy", float_fmt="{:.3f}")
print("Ratios near 1.0 mean the legacy figure is the time to PRINT the results, not to classify the sample.")

## 5 · Flash and memory

In [ ]:
flash_rows = []
FLASH_MAX = 1_310_720   # default 'Default 4MB with spiffs' app partition
for _, r in leg.iterrows():
    flash_rows.append({"dataset": r["dataset"], "firmware": "old serial-print benchmark (incl. embedded test vectors)", "bytes": r["program_storage_bytes"]})
for ds, log in timed.items():
    b = log["meta"].get("sketch_size_bytes")
    if b: flash_rows.append({"dataset": ds, "firmware": "benchmark sketch, exact export (incl. embedded test vectors)", "bytes": int(b)})
# model-only footprint = (firmware with model) - (firmware without model)
for ds in DATASETS:
    a, b = LOGS / f"{ds}_footprint_with_model.txt", LOGS / f"{ds}_footprint_without_model.txt"
    if a.exists() and b.exists():
        sa, sb = parse_footprint(a), parse_footprint(b)
        if sa and sb: flash_rows.append({"dataset": ds, "firmware": "MODEL ONLY (with − without)", "bytes": sa - sb})
fl = pd.DataFrame(flash_rows)
if len(fl):
    fl["% of 1.25 MB app partition"] = (fl["bytes"] / FLASH_MAX * 100).round(1)
    fl["dataset"] = fl["dataset"].map(DS_LABEL)
    display(fl)
    save_table(fl, TAB / "hw_flash", caption="Flash usage on the ESP32 (bytes).", label="tab:hw_flash", float_fmt="{:.1f}")
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    piv = fl.pivot_table(index="dataset", columns="firmware", values="bytes") / 1024
    piv.plot.bar(ax=ax, rot=0); ax.axhline(FLASH_MAX / 1024, color="red", ls="--", lw=1); ax.text(0, FLASH_MAX / 1024 * .95, "app partition limit", color="red", fontsize=7, va="top")
    ax.set_ylabel("KB"); ax.set_xlabel(""); ax.legend(fontsize=6)
    ax.set_title("ESP32 flash usage")
    save_fig(fig, FIG / "hw_flash"); plt.show()
if timed:
    mem = hw[hw.dataset.isin(timed)][[c for c in ["dataset", "heap_size_bytes", "free_heap_before", "free_heap_after", "min_free_heap"] if c in hw.columns]]
    display(mem); save_table(mem, TAB / "hw_memory", caption="ESP32 heap measurements (bytes).", label="tab:hw_mem", float_fmt="{:.0f}")

## 6 · Final hardware summary table

In [ ]:
if len(hw):
    show = hw.copy(); show["dataset"] = show["dataset"].map(DS_LABEL)
    display(show.T)
    show.to_csv(TAB / "hw_summary_full.csv", index=False)
    json.dump({ds: summarize_hardware(ds, l) for ds, l in parsed.items()}, open(ROOT / "esp32" / "hardware_summary.json", "w"), indent=2, default=str)
    print("saved paper/tables/hw_summary_full.csv and esp32/hardware_summary.json")